# 01 — Rouwenhorst Markov chain for the labour productivity process

**Ports**: `Code/MATLAB/Exogenous_files/rouwenhorst.m` and the post-processing of `params.ex` / `params.piex` in `Code/MATLAB/Main.m` lines 45–60.

**Goal**: build a 7-state Markov chain that approximates the AR(1) labour productivity process used by de Ferra, Mitman, and Romei (2020) ($\rho_l = 0.97$, $\sigma_l = 0.84$ cross-sectional std → innovation std $\sigma_\varepsilon = 0.84\sqrt{1-\rho^2}$).

**Outputs (saved to `../output/markov.npz`)**:
- `ex` — 7-vector of labour productivity values, normalised so that $\bar{\ell}\cdot\bar{e} = 1$.
- `piex` — 7×7 transition matrix.
- `exinv` — ergodic distribution.
- `ex_mean` — should equal 1 after normalisation.

## Imports

In [1]:
from pathlib import Path

import numpy as np

OUTPUT_DIR = Path('..') / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Rouwenhorst function

Port of `Code/MATLAB/Exogenous_files/rouwenhorst.m` (Floden, 2010).

Approximates the AR(1):
$$z_{t+1} = (1-\rho)\mu + \rho z_t + \varepsilon_{t+1},\quad \varepsilon_{t+1}\sim\mathcal N(0,\sigma^2)$$
by an $N$-state Markov chain whose conditional and unconditional 1st/2nd moments match the AR(1) exactly (Kopecky & Suen, 2010).

In [2]:
def rouwenhorst(N, mu, rho, sigma):
    """Rouwenhorst (1995) discretisation of an AR(1).

    Parameters
    ----------
    N : int
        Number of grid nodes.
    mu : float
        Unconditional mean of the AR(1).
    rho : float
        Persistence.
    sigma : float
        Standard deviation of the *innovations* (not the cross-section).

    Returns
    -------
    Z : (N,) ndarray
        Grid of states.
    PI : (N, N) ndarray
        Row-stochastic transition matrix; PI[i, j] = P(z' = j | z = i).
    """
    sigmaz = sigma / np.sqrt(1 - rho ** 2)
    p = (1 + rho) / 2

    PI = np.array([[p, 1 - p], [1 - p, p]])

    for n in range(3, N + 1):
        zeros_col = np.zeros((n - 1, 1))
        zeros_row = np.zeros((1, n))

        PI_top_left = np.block([[PI, zeros_col], [zeros_row]])
        PI_top_right = np.block([[zeros_col, PI], [zeros_row]])
        PI_bot_left = np.block([[zeros_row], [PI, zeros_col]])
        PI_bot_right = np.block([[zeros_row], [zeros_col, PI]])

        PI = (
            p * PI_top_left
            + (1 - p) * PI_top_right
            + (1 - p) * PI_bot_left
            + p * PI_bot_right
        )
        PI[1:-1, :] /= 2  # internal rows are double-counted; matches MATLAB line 41

    fi = np.sqrt(N - 1) * sigmaz
    Z = np.linspace(-fi, fi, N) + mu
    return Z, PI

## Build the chain with the paper's calibration

From `Main.m` lines 46–53:
```matlab
params.nx   = 7;
params.rho  = .97;
sigy        = .84*sqrt(1-params.rho^2);
[params.ex, params.piex] = rouwenhorst(params.nx, 0, rho, sigy);
```

In [3]:
nx = 7
rho = 0.97
sigma_cross = 0.84  # cross-sectional std of log productivity (paper Table 1)
sigy = sigma_cross * np.sqrt(1 - rho ** 2)  # innovation std

Z, PI = rouwenhorst(nx, 0.0, rho, sigy)
print('Z (log grid):', Z)
print('Row sums of PI:', PI.sum(axis=1))  # should all be 1.0

Z (log grid): [-2.05757138 -1.37171426 -0.68585713  0.          0.68585713  1.37171426
  2.05757138]
Row sums of PI: [1. 1. 1. 1. 1. 1. 1.]


## Post-processing: exponentiate and normalise mean to 1

From `Main.m` lines 54–60:
```matlab
params.ex    = exp(params.ex);
temp         = params.piex^1000000;
params.exinv = temp(1,:);
params.ex    = params.ex/(params.exinv*params.ex);
params.ex_mean = params.exinv*params.ex;
```

Instead of taking the millionth matrix power, we use `scipy.linalg.eig` to get the ergodic distribution directly (numerically equivalent, faster, more robust).

In [4]:
ex = np.exp(Z)

eigvals, eigvecs = np.linalg.eig(PI.T)
ergodic_idx = np.argmin(np.abs(eigvals - 1.0))
exinv = np.real(eigvecs[:, ergodic_idx])
exinv = exinv / exinv.sum()

ex = ex / (exinv @ ex)
ex_mean = exinv @ ex

print('Productivity grid (normalised):', ex)
print('Ergodic distribution:', exinv)
print('Mean (must be ~1.0):', ex_mean)

assert np.isclose(ex_mean, 1.0, atol=1e-12), 'ex_mean should be exactly 1 after normalisation'
assert np.allclose(exinv @ PI, exinv, atol=1e-12), 'exinv should be invariant under PI'

Productivity grid (normalised): [0.090386   0.17945895 0.35631089 0.70744562 1.40461412 2.78882328
 5.53713307]
Ergodic distribution: [0.015625 0.09375  0.234375 0.3125   0.234375 0.09375  0.015625]
Mean (must be ~1.0): 0.9999999999999999


## Sanity checks against the paper / MATLAB

- The cross-sectional std of $\log e$ should equal $\sigma_l = 0.84$.
- The persistence (one-step autocorrelation in levels of $\log e$) should equal $\rho = 0.97$.

In [5]:
log_ex = np.log(ex)
implied_mean_log = exinv @ log_ex
log_ex_centered = log_ex - implied_mean_log
implied_var_log = exinv @ log_ex_centered ** 2
implied_std_log = np.sqrt(implied_var_log)

# Cov(z_t, z_{t+1}) = sum_i pi_i (z_i - mu) * E[z' - mu | z_i]
cond_exp_centered = PI @ log_ex_centered
implied_cov = exinv @ (log_ex_centered * cond_exp_centered)
implied_autocorr = implied_cov / implied_var_log

print(f'Implied cross-sectional std of log e : {implied_std_log:.6f}  (target 0.84)')
print(f'Implied 1-step autocorrelation       : {implied_autocorr:.6f}  (target 0.97)')

# Rouwenhorst exactly matches the AR(1) moments, so these should be tight.
assert np.isclose(implied_std_log, sigma_cross, atol=1e-10)
assert np.isclose(implied_autocorr, rho, atol=1e-10)

Implied cross-sectional std of log e : 0.840000  (target 0.84)
Implied 1-step autocorrelation       : 0.970000  (target 0.97)


## Save outputs

In [6]:
out_path = OUTPUT_DIR / 'markov.npz'
np.savez(
    out_path,
    ex=ex,
    piex=PI,
    exinv=exinv,
    ex_mean=ex_mean,
    Z_log=Z,
    rho=rho,
    sigma_cross=sigma_cross,
    sigma_innov=sigy,
    nx=nx,
)
print(f'Saved: {out_path.resolve()}')

Saved: /Users/siyingli/github/de-Ferra2020-kz/Code/Python/output/markov.npz
